# Passo 1 - Importações e Configurações

In [1]:
import os
import io
import psycopg2
import pandas as pd
import numpy as np
from datetime import datetime
from minio import Minio
from dotenv import load_dotenv

load_dotenv()

# --- Configurações MinIO ---
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'localhost:9000').replace('http://', '').replace('https://', '')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minio@1234!')
BUCKET           = 'moedas'
CAMADA_ORIGEM    = 'silver'
CAMADA_DESTINO   = 'gold'

# --- Configurações PostgreSQL Data ---
PG_CONFIG = {
    "dbname": os.getenv("DB_NAME", "moedas"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5434")
}

def get_minio_client():
    return Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=False
    )

print("Configurações da Camada Gold carregadas!")

Configurações da Camada Gold carregadas!


# Passo 2 - Funções de Conexão e Auxiliares

In [2]:
def salvar_parquet_minio(df: pd.DataFrame, bucket: str, object_key: str):
    """Salva DataFrame como Parquet no MinIO."""
    client = get_minio_client()
    if not client.bucket_exists(bucket):
        client.make_bucket(bucket)

    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine='pyarrow')
    buf.seek(0)
    tamanho = buf.getbuffer().nbytes

    client.put_object(
        bucket_name=bucket,
        object_name=object_key,
        data=buf,
        length=tamanho,
        content_type='application/octet-stream'
    )
    print(f"   -> MinIO: s3://{bucket}/{object_key} ({tamanho / 1024:.1f} KB)")

# Testa conexão com Postgres
try:
    conn = psycopg2.connect(**PG_CONFIG)
    conn.close()
    print(" Conexão com PostgreSQL Data OK!")
except Exception as e:
    print(f" Erro na conexão com PostgreSQL: {e}")

 Conexão com PostgreSQL Data OK!


# Passo 3 - Leitura dos Dados Limpos da Camada Silver

In [3]:
def ler_silver_postgres() -> pd.DataFrame:
    """Lê a tabela unificada da Camada Silver no Postgres."""
    conn = psycopg2.connect(**PG_CONFIG)
    query = """
        SELECT 
            id_extracao, codigo_moeda, nome_moeda, valor_compra, valor_venda,
            spread_venda_compra, alta, baixa, amplitude_diaria, variacao,
            pct_mudanca, data_cotacao, data_atualizacao, ano, mes, dia, hora
        FROM public.extracao_moedas_silver;
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df_silver = ler_silver_postgres()
print(f"Registros lidos da Silver: {len(df_silver):,}")
df_silver.head(3)

Registros lidos da Silver: 66


C:\Users\Admin\AppData\Local\Temp\ipykernel_6852\2009269094.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id_extracao,codigo_moeda,nome_moeda,valor_compra,valor_venda,spread_venda_compra,alta,baixa,amplitude_diaria,variacao,pct_mudanca,data_cotacao,data_atualizacao,ano,mes,dia,hora
0,20260827013050,USD,Dólar Americano/Real Brasileiro,5.1487,5.1500,0.0013,5.1641,5.1402,0.0239,0.0015,0.0291,2026-08-26 19:30:14,2026-08-27 01:30:50.756969,2026,8,27,1
1,20260827013050,EUR,Euro/Real Brasileiro,5.9952,6.0096,0.0144,6.0181,5.9844,0.0337,-0.0133,-0.2214,2026-08-26 19:31:25,2026-08-27 01:30:50.756969,2026,8,27,1
2,20260827013050,BTC,Bitcoin/Real Brasileiro,406542.0000,406543.0000,1.0000,409102.0000,401702.0000,7400.0000,-582.0000,-0.1430,2026-08-26 22:09:28,2026-08-27 01:30:50.756969,2026,8,27,1


# Passo 4 - Regras de Negócio Gold 1: Resumo Diário (gold_cotacoes_diarias)
Consolida valor médio, mínimo, máximo, preço de fechamento do dia e volatilidade por moeda.

In [4]:
# Ordena por data para garantir ordenação temporal
df_sorted = df_silver.sort_values(by=['codigo_moeda', 'data_atualizacao'])

# Agrupa por dia e moeda
df_gold_diario = (
    df_sorted.groupby(['codigo_moeda', 'nome_moeda', 'ano', 'mes', 'dia'])
    .agg(
        qtd_coletas=('id_extracao', 'count'),
        preco_abertura=('valor_compra', 'first'),
        preco_fechamento=('valor_compra', 'last'),
        preco_medio=('valor_compra', 'mean'),
        preco_minimo=('valor_compra', 'min'),
        preco_maximo=('valor_compra', 'max'),
        spread_medio=('spread_venda_compra', 'mean'),
        variacao_percentual_dia=('pct_mudanca', 'last'),
        ultima_atualizacao=('data_atualizacao', 'max')
    )
    .reset_index()
)

# Arredondamento numérico
cols_round = ['preco_abertura', 'preco_fechamento', 'preco_medio', 'preco_minimo', 'preco_maximo', 'spread_medio']
df_gold_diario[cols_round] = df_gold_diario[cols_round].round(4)

print(f"Tabela 'gold_cotacoes_diarias' gerada com {len(df_gold_diario):,} linhas!")
df_gold_diario.head()

Tabela 'gold_cotacoes_diarias' gerada com 9 linhas!


,codigo_moeda,nome_moeda,ano,mes,dia,qtd_coletas,preco_abertura,preco_fechamento,preco_medio,preco_minimo,preco_maximo,spread_medio,variacao_percentual_dia,ultima_atualizacao
0,BTC,Bitcoin/Real Brasileiro,2026,8,27,11,406542.0000,416987.0000,411843.0909,406542.0000,417126.0000,3.1818,2.8470,2026-08-27 18:00:02.221643
1,BTC,Bitcoin/Real Brasileiro,2026,8,28,6,410728.0000,407136.0000,409135.6667,405651.0000,412454.0000,9.6667,-2.3640,2026-08-28 18:00:01.888667
2,BTC,Bitcoin/Real Brasileiro,2026,8,29,5,405180.0000,404051.0000,404951.2000,404051.0000,405994.0000,1.0000,-2.0150,2026-08-29 12:00:01.728180
3,EUR,Euro/Real Brasileiro,2026,8,27,11,5.9952,6.0162,6.0052,5.9952,6.0251,0.0088,0.3503,2026-08-27 18:00:02.221643
4,EUR,Euro/Real Brasileiro,2026,8,28,6,6.0106,6.0367,6.0276,6.0106,6.0560,0.0020,0.3558,2026-08-28 18:00:01.888667


# Passo 5 - Regras de Negócio Gold 2: Médias Móveis (gold_medias_moveis)
Calcula tendências de mercado com médias móveis simples (3, 7 e 14 períodos) para apoiar gráficos de linha.

In [5]:
df_mm = df_sorted.copy()

# Calcula médias móveis por moeda
df_mm['media_movel_3p'] = df_mm.groupby('codigo_moeda')['valor_compra'].transform(lambda x: x.rolling(3, min_periods=1).mean()).round(4)
df_mm['media_movel_7p'] = df_mm.groupby('codigo_moeda')['valor_compra'].transform(lambda x: x.rolling(7, min_periods=1).mean()).round(4)
df_mm['media_movel_14p'] = df_mm.groupby('codigo_moeda')['valor_compra'].transform(lambda x: x.rolling(14, min_periods=1).mean()).round(4)

colunas_mm = [
    'id_extracao', 'codigo_moeda', 'nome_moeda', 'valor_compra',
    'media_movel_3p', 'media_movel_7p', 'media_movel_14p',
    'pct_mudanca', 'data_atualizacao'
]

df_gold_medias_moveis = df_mm[colunas_mm]
print(f"Tabela 'gold_medias_moveis' gerada com {len(df_gold_medias_moveis):,} linhas!")
df_gold_medias_moveis.head()

Tabela 'gold_medias_moveis' gerada com 66 linhas!


,id_extracao,codigo_moeda,nome_moeda,valor_compra,media_movel_3p,media_movel_7p,media_movel_14p,pct_mudanca,data_atualizacao
2,20260827013050,BTC,Bitcoin/Real Brasileiro,406542.0,406542.0000,406542.0,406542.0,-0.143,2026-08-27 01:30:50.756969
5,20260827014027,BTC,Bitcoin/Real Brasileiro,407760.0,407151.0000,407151.0,407151.0,0.269,2026-08-27 01:40:27.372228
8,20260827020000,BTC,Bitcoin/Real Brasileiro,407619.0,407307.0000,407307.0,407307.0,0.224,2026-08-27 02:00:00.863800
11,20260827102744,BTC,Bitcoin/Real Brasileiro,410587.0,408655.3333,408127.0,408127.0,1.240,2026-08-27 10:27:44.704907
14,20260827110001,BTC,Bitcoin/Real Brasileiro,411071.0,409759.0000,408715.8,408715.8,1.275,2026-08-27 11:00:01.679690


# Passo 6 - Regras de Negócio Gold 3: Snapshot de Cotação Atual (gold_kpi_atual)
Pega apenas a última cotação registrada de cada moeda para uso em cards numéricos no Metabase.

In [6]:
df_gold_kpi_atual = (
    df_sorted.sort_values('data_atualizacao')
    .groupby('codigo_moeda')
    .last()
    .reset_index()
)

colunas_kpi = [
    'codigo_moeda', 'nome_moeda', 'valor_compra', 'valor_venda',
    'spread_venda_compra', 'alta', 'baixa', 'pct_mudanca', 'data_atualizacao'
]

df_gold_kpi_atual = df_gold_kpi_atual[colunas_kpi]
print(f"Tabela 'gold_kpi_atual' gerada com {len(df_gold_kpi_atual):,} moedas!")
df_gold_kpi_atual

Tabela 'gold_kpi_atual' gerada com 3 moedas!


,codigo_moeda,nome_moeda,valor_compra,valor_venda,spread_venda_compra,alta,baixa,pct_mudanca,data_atualizacao
0,BTC,Bitcoin/Real Brasileiro,404051.0000,404052.0000,1.000,416221.0000,402496.000,-2.0150,2026-08-29 12:00:01.728180
1,EUR,Euro/Real Brasileiro,6.0056,6.0076,0.002,6.0561,5.987,-0.1613,2026-08-29 12:00:01.728180
2,USD,Dólar Americano/Real Brasileiro,5.1850,5.1860,0.001,5.2282,5.159,0.4436,2026-08-29 12:00:01.728180


# Passo 7 - Salvamento das Tabelas no MinIO e PostgreSQL Gold

In [7]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. Salva no MinIO (Formato Parquet)
print("Persistindo arquivos Parquet no MinIO (Camada Gold)...")
salvar_parquet_minio(df_gold_diario, BUCKET, f"{CAMADA_DESTINO}/cotacoes_diarias/cotacoes_diarias_{timestamp}.parquet")
salvar_parquet_minio(df_gold_medias_moveis, BUCKET, f"{CAMADA_DESTINO}/medias_moveis/medias_moveis_{timestamp}.parquet")
salvar_parquet_minio(df_gold_kpi_atual, BUCKET, f"{CAMADA_DESTINO}/kpi_atual/kpi_atual_{timestamp}.parquet")

# 2. Persiste no PostgreSQL (Data Lake / Metabase)
def salvar_postgres_gold(df: pd.DataFrame, nome_tabela: str):
    conn = psycopg2.connect(**PG_CONFIG)
    cursor = conn.cursor()
    
    # Grava substituindo ou criando a tabela no Postgres
    from sqlalchemy import create_engine
    engine = create_engine(f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['dbname']}")
    df.to_sql(nome_tabela, engine, schema='public', if_exists='replace', index=False)
    
    conn.close()
    print(f"   -> Tabela PostgreSQL 'public.{nome_tabela}' atualizada com sucesso!")

print("\nAtualizando tabelas Gold no PostgreSQL...")
salvar_postgres_gold(df_gold_diario, "gold_cotacoes_diarias")
salvar_postgres_gold(df_gold_medias_moveis, "gold_medias_moveis")
salvar_postgres_gold(df_gold_kpi_atual, "gold_kpi_atual")

print("\nPIPELINE GOLD FINALIZADO COM SUCESSO!")

Persistindo arquivos Parquet no MinIO (Camada Gold)...
   -> MinIO: s3://moedas/gold/cotacoes_diarias/cotacoes_diarias_20260829_095726.parquet (9.3 KB)
   -> MinIO: s3://moedas/gold/medias_moveis/medias_moveis_20260829_095726.parquet (8.5 KB)
   -> MinIO: s3://moedas/gold/kpi_atual/kpi_atual_20260829_095726.parquet (6.0 KB)

Atualizando tabelas Gold no PostgreSQL...
   -> Tabela PostgreSQL 'public.gold_cotacoes_diarias' atualizada com sucesso!
   -> Tabela PostgreSQL 'public.gold_medias_moveis' atualizada com sucesso!
   -> Tabela PostgreSQL 'public.gold_kpi_atual' atualizada com sucesso!

PIPELINE GOLD FINALIZADO COM SUCESSO!


# Passo 8 - Auditoria e Listagem dos Arquivos na Gold

In [9]:
client = get_minio_client()
objetos = list(client.list_objects(BUCKET, prefix='gold/', recursive=True))

print(f"[{BUCKET}] Camada Gold — {len(objetos)} arquivo(s) Parquet gerado(s):\n")
for obj in sorted(objetos, key=lambda x: x.object_name):
    print(f"{obj.object_name:<80} {obj.size / 1024:>7.1f} KB")

[moedas] Camada Gold — 3 arquivo(s) Parquet gerado(s):

gold/cotacoes_diarias/cotacoes_diarias_20260829_095726.parquet                       9.3 KB
gold/kpi_atual/kpi_atual_20260829_095726.parquet                                     6.0 KB
gold/medias_moveis/medias_moveis_20260829_095726.parquet                             8.5 KB
